# components/file_browser_panel

> File browser panel configuration and rendering for browsing local audio/video files

In [ ]:
#| default_exp components.file_browser_panel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional, Callable
from pathlib import Path

from fasthtml.common import Div

from cjm_fasthtml_file_browser.core.config import (
    FileBrowserConfig, FilterConfig, ViewConfig, SelectionMode, FileColumn
)
from cjm_fasthtml_file_browser.core.models import BrowserState, BrowserSelection
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider

from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_direction, grow
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds

## Media Extensions

Audio and video extensions in dot-less format for `FilterConfig`.

In [ ]:
#| export
AUDIO_FILTER_EXTENSIONS = ["mp3", "wav", "flac", "aac", "ogg", "m4a", "wma", "opus"]
VIDEO_FILTER_EXTENSIONS = ["mp4", "mkv", "avi", "mov", "webm", "wmv", "flv"]
MEDIA_FILTER_EXTENSIONS = AUDIO_FILTER_EXTENSIONS + VIDEO_FILTER_EXTENSIONS

## Browser Configuration

Creates a `FileBrowserConfig` for audio/video file selection. Uses the `cjm-fasthtml-file-browser` library's built-in selection UI.

In [ ]:
#| export
def create_media_browser_config() -> FileBrowserConfig:  # Configured for audio/video file selection
    """Create file browser config for audio/video file selection with folder bulk-select."""
    return FileBrowserConfig(
        selection_mode=SelectionMode.MULTIPLE,
        selectable_types="both",
        view=ViewConfig(
            columns=[FileColumn.NAME, FileColumn.SIZE, FileColumn.MODIFIED],
            sort_folders_first=True,
        ),
        filter=FilterConfig(
            allowed_extensions=MEDIA_FILTER_EXTENSIONS,
            show_directories=True,
            show_hidden=True,
        ),
        show_path_bar=True,
        show_path_input=True,
        container_id=SourceSelectHtmlIds.BROWSER_PANEL,
        content_id=SourceSelectHtmlIds.BROWSER_FILE_LIST,
        vc_prefix="tss_fb",
    )

## Browser State

Helpers to create and restore `BrowserState` from step state.

In [ ]:
#| export
def get_browser_state(
    step_state: Dict[str, Any],  # Current step state
    default_path: str = "",  # Default directory if no state exists
) -> BrowserState:  # Browser state for the file browser
    """Get or create BrowserState from step state."""
    state_dict = step_state.get("browser_state", {})
    if state_dict:
        return BrowserState.from_dict(state_dict)
    return BrowserState(current_path=default_path or str(Path.home()))

In [ ]:
#| export
def sync_browser_selection(
    browser_state: BrowserState,  # Browser state to update
    selected_files: List[Dict[str, Any]],  # Current selected_files from step state
    selected_folders: Optional[List[str]] = None,  # Folder paths toggled for bulk select
) -> None:  # Modifies browser_state in place
    """Sync browser selection state with selected files and folders."""
    paths = [f["path"] for f in selected_files]
    if selected_folders:
        paths.extend(selected_folders)
    browser_state.selection.selected_paths = paths

## Render Browser Panel

Wraps the file browser library's `FileBrowserRouters.render` callable in a flex container for proper height constraint propagation.

In [ ]:
#| export
def render_browser_panel(
    render_fn: Callable,  # FileBrowserRouters.render callable
) -> Any:  # Rendered file browser in a flex container
    """Render the file browser panel using the library's self-contained VC component."""
    return Div(
        render_fn(),
        id=SourceSelectHtmlIds.BROWSER_PANEL,
        cls=combine_classes(
            w.full, min_h(0),
            flex_display, flex_direction.col, grow(),
        ),
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()